# Step 1 : PDF to markdown

**NOTE : YOU SHOULD HAVE A .env FILE WITH YOUR AZURE API KEY**

In [ ]:
import os
import base64
from pathlib import Path
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentContentFormat
from dotenv import load_dotenv

load_dotenv()  # load AZURE_ENDPOINT and AZURE_KEY if present
BASE_DIR = Path.cwd().parent

def pdf_to_markdown_pages(
    pdf_path: str | Path,
    *,
    endpoint: str | None = None,
    key: str | None = None,
) -> list[str]:
    """Return one Markdown string per PDF page using Azure Document Intelligence."""
    # 1) Resolve credentials
    endpoint = endpoint or os.getenv("AZURE_ENDPOINT")
    key = key or os.getenv("AZURE_KEY")
    if not endpoint or not key:
        raise ValueError("Missing AZURE_ENDPOINT or AZURE_KEY.")

    # 2) Create client
    client = DocumentIntelligenceClient(endpoint, AzureKeyCredential(key))

    # 3) Read file and start analysis (prebuilt layout → Markdown)
    pdf_bytes = Path(pdf_path).read_bytes()
    poller = client.begin_analyze_document(
        model_id="prebuilt-layout",
        body={"base64Source": base64.b64encode(pdf_bytes).decode()},
        output_content_format=DocumentContentFormat.MARKDOWN,
    )

    # 4) Wait for result
    result = poller.result()

    # 5) Split full Markdown into per-page slices using span offsets
    full_md = result.content or ""
    pages_md: list[str] = []
    for page in sorted(result.pages, key=lambda p: p.page_number):
        if not page.spans:
            pages_md.append("")
            continue
        start = min(s.offset for s in page.spans)
        end = max(s.offset + s.length for s in page.spans)
        pages_md.append(full_md[start:end])

    return pages_md
     

In [ ]:
example_pdf = BASE_DIR / "files/raw_cases/proof_of_service_example.pdf"

pages = pdf_to_markdown_pages(example_pdf)

# Save the result as a Markdown file
output_path = BASE_DIR / "files/parsed_cases/proof_of_service_example.md"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    for i, md in enumerate(pages, start=1):
        f.write(f"\n{'='*60}\n")
        f.write(f"  PAGE {i}\n")
        f.write(f"{'='*60}\n\n")
        f.write(md)

print(f"Saved to {output_path}")

# Step 2 : Extract information from markdown

**NOTE : YOU SHOULD HAVE A .env FILE WITH YOUR OPENAI API KEY**

In [7]:
from typing import Optional, Any, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph


# --- Load the parsed markdown
md_content = (BASE_DIR / "files/parsed_cases/proof_of_service_example.md").read_text(encoding="utf-8")

# --- LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- Prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a legal document analyst. Answer based only on the document provided."),
    ("human", "Document:\n{document}\n\nQuestion: {question}"),
])

chain = prompt | llm


# --- State
class QAState(TypedDict, total=False):
    pos_type: Optional[Any]
    attorney_fees: Optional[Any]


# --- Async nodes
async def start(state: QAState) -> QAState:
    return state


async def ask_pos_type(state: QAState) -> QAState:
    res = await chain.ainvoke({
        "document": md_content,
        "question": "What type of proof of service is this? (personal, substituted, mail, etc.) Answer with just the type.",
    })
    return {"pos_type": res.content}


async def ask_attorney_fees(state: QAState) -> QAState:
    res = await chain.ainvoke({
        "document": md_content,
        "question": "What are the attorney fees? Return only the integer amount in dollars, nothing else.",
    })
    return {"attorney_fees": int(res.content.replace("$", "").replace(",", "").strip())}


# --- Parallel graph
builder = StateGraph(state_schema=QAState)
builder.add_node("start", start)
builder.add_node("ask_pos_type", ask_pos_type)
builder.add_node("ask_attorney_fees", ask_attorney_fees)

builder.set_entry_point("start")
builder.add_edge("start", "ask_pos_type")       # fan-out
builder.add_edge("start", "ask_attorney_fees")   # fan-out

graph = builder.compile()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/othmanbensouda/Desktop/proof_of_service/code/files/parsed_cases/proof_of_service_example.md'

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
async def main():
    out = await graph.ainvoke({})
    print("Type of POS:", out["pos_type"])
    print("Attorney fees:", out["attorney_fees"])

await main()